# Balance, Memory & Coherence

**Purpose:**
Explore how different assumptions about memory and balance shape system behavior, using simplw numerical operators inspired by state-space dynamics.

**Question:**
How do responsiveness and stability trade off as we vary how a state remembers its past?



In [10]:
import numpy as np
import pandas as pd

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [20]:
# reincarnating Day 1 synthetic signal data:

rng = np.random.default_rng(7) #generate a random sample from a given array

n_minutes = 24 * 60
t = pd.date_range('2026-01-01', periods=n_minutes, freq='min')

sensor_C = (
    5
    + 0.002 * np.arange(n_minutes)
    + rng.normal(0, 0.15, size=n_minutes)
)

df = pd.DataFrame({'vib_C': sensor_C}, index=t)

#Inject chaos - missingness and anomaly burst
missing_idx = rng.choice(n_minutes, size=60, replace=False)
df.iloc[missing_idx, df.columns.get_loc('vib_C')] = np.nan

burst_start = 600
df.iloc[burst_start:burst_start + 30, df.columns.get_loc('vib_C')] += 1.5

# Signal used by models
v = df['vib_C'].interpolate(limit_direction='both').to_numpy()
n = len(v)

In [30]:
def kalman_filter_1d(y, Q, R, x0=0.0, P0=1.0):
    """

    1D Kalman for for:
        x_t = x_{t-1} + w_t,    w_t ~ N(0, Q)
        y_t = x_t + v_t,        v_t ~ N(0, R)

    Returns:
        x_filt: filtered state means
        p_filt: filtered state variances
        innovationsL prediction residuals
        S_valsL innovation variances
    """

    n = len(y)
    x_filt = np.zeros(n)
    P_filt = np.zeros(n)
    innovations = np.zeros(n)
    S_vals = np.zeros(n)

    x_prev = x0
    P_prev = P0

    for t in range(n):
        # Predict
        x_pred = x_prev
        P_pred = P_prev + Q

        # Innovation
        innov = y[t] - x_pred
        S = P_pred + R

        #Update
        K = P_pred / S
        x_new = x_pred + K * innov
        P_new = (1-K) * P_pred

        x_filt[t] = x_new
        P_filt[t] = innov
        S_vals[t] = S

        x_prev, P_prev = x_new, P_new

    return x_filt, P_filt, innovations, S_vals

## Assumptions going in

- What I believe about how the system behaves:
  -

- What I expect to happen:
  -

In [31]:
regimes = {
    'rigid+memory': {'Q': 0.0005**2, 'R': 0.15**2},
    'balanced': {'Q': 0.002**2, 'R': 0.15**2},
    'reactive': {'Q': 0.01**2, 'R': 0.15**2},    
}
regimes

{'rigid+memory': {'Q': 2.5e-07, 'R': 0.0225},
 'balanced': {'Q': 4e-06, 'R': 0.0225},
 'reactive': {'Q': 0.0001, 'R': 0.0225}}

In [32]:
results = {}

for name, params in regimes.items():
    x_hat, P_hat, innov, S = kalman_filter_1d(
        y=v,
        Q=params['Q'],
        R=params['R'],
        x0=v[0],
        P0=1.0
    )
    z_innov = innov / np.sqrt(S)
    results[name] = {'x_hat': x_hat, 'P_hat': P_hat, 'z_innov': z_innov}


In [38]:
closure_metrics = {}

for name, res in results.items():
    x_hat = res['x_hat']
    P_hat = res['P_hat']
    z = res['z_innov']
    print(name, "min(P_hat) =", np.min(P_hat), "  count<0 =", np.sum(P_hat < 0))

   # dx = np.abs(np.diff(x_hat))
    #dP = np.abs(np.diff(P_hat))

    P_safe = np.maximum(P_hat, 0.0)

    closure_metrics[name] = {
        'mean_abs_state_change': float(np.mean(dx)),
        'final_state_change': float(np.mean(dx[-50:])),
        'mean_uncertainty': float(np.mean(np.sqrt(P_safe))),
        'final_uncertainty': float(np.mean(np.sqrt(P_safe[-50:]))),
        'innov_std': float(np.std(z)),
    }

closure_metrics

rigid+memory min(P_hat) = -0.29861831096590397   count<0 = 49
balanced min(P_hat) = -0.5052509097812123   count<0 = 320
reactive min(P_hat) = -1.2613149607028262   count<0 = 609


{'rigid+memory': {'mean_abs_state_change': 0.009182296765491822,
  'final_state_change': 0.00976341911348145,
  'mean_uncertainty': 0.6223102521031671,
  'final_uncertainty': 0.7612109814456592,
  'innov_std': 0.0},
 'balanced': {'mean_abs_state_change': 0.009182296765491822,
  'final_state_change': 0.00976341911348145,
  'mean_uncertainty': 0.32643606453299945,
  'final_uncertainty': 0.3539924171075961,
  'innov_std': 0.0},
 'reactive': {'mean_abs_state_change': 0.009182296765491822,
  'final_state_change': 0.00976341911348145,
  'mean_uncertainty': 0.20029319587507188,
  'final_uncertainty': 0.20151870404577038,
  'innov_std': 0.0}}

## Reflection

This exercise showed that coherence emerges from balance, not extremes. When assumptions were well matched to the signal, state estimates stabilized and uncertainty settled. When assumptions were mismatched, the system either resisted adaptation or failed to close. Small numerical issues reinforced that even theoretically well-behaved models require practical safeguards in implementation.

Closure emerged most clearly in the balanced regime, where state updates dereased over time and uncertainty stabilized. When memory dominated, the system resisted change but failed to adapt to genuine shifts. When responsiveness dominated, the estimate remained volatile and never fully settled.

This illustrates that cogherence is not the absence o change, but the stabilization of interpretation under continued observation